# PVAMU Energy Trading Program

## Python Workshop: Core pandas Practice with ERCOT RTM Hub Price Data

In this exercise, we will use **real ERCOT Real-Time Market (RTM) hub price data** to practice core **pandas** skills.

By the end of this notebook, we will build a **12 x 24 pivot table** showing the **average Houston Hub price** for each **month** and **delivery hour**.

Along the way, we will practice how to:

* load a CSV into a DataFrame
* inspect the structure of a DataFrame
* convert a column to **datetime**
* create a new column from a datetime column
* filter data with a **Boolean mask**
* summarize data with **groupby**
* reshape data with a **pivot table**


## Step 1: Load the Data

Before we can analyze anything, we first need to load our dataset into Python as a **dataframe**.

In this step, we will:

* import the **pandas** library
* use `pd.read_csv()` to read the CSV file
* store the dataset in a variable called `df`
* use `df.head()` to display the first few rows so we can confirm the file loaded correctly

This is usually the first step in any pandas workflow.


In [1]:
# Import pandas
import pandas as pd

# Read the csv and turn it into a DataFrame
df = pd.read_csv('ercot_rtm_Hub_prices_2025.csv')

# Display first few rows
df.head()

,Delivery Date,Delivery Hour,Settlement Point Name,Average Hourly Price
0,2025-01-01,1,HB_HOUSTON,18.0775
1,2025-01-01,1,HB_NORTH,18.0775
2,2025-01-01,1,HB_PAN,18.0775
3,2025-01-01,1,HB_SOUTH,18.0775
4,2025-01-01,1,HB_WEST,18.0775


## Step 2: Data Exploration

Now that the data is loaded, the next step is to quickly inspect it.

Some of the most useful beginner commands are:

* `df.shape` → shows the number of rows and columns
* `df.dtypes` → shows the data type of each column
* `df.info()` → gives a quick summary of the DataFrame. Useful for checking missing values (missing data)
  
These commands help us understand what kind of data we are working with before we start cleaning or analyzing it.

In [2]:
# Show the "shape" of the DataFrame
df.shape

(43795, 4)

In [3]:
# Show the data types of each column
df.dtypes

Delivery Date             object
Delivery Hour              int64
Settlement Point Name     object
Average Hourly Price     float64
dtype: object

In [4]:
# Check for missing data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43795 entries, 0 to 43794
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Delivery Date          43795 non-null  object 
 1   Delivery Hour          43795 non-null  int64  
 2   Settlement Point Name  43795 non-null  object 
 3   Average Hourly Price   43795 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.3+ MB


## Step 3: Data Cleaning 

Right now, the `Delivery Date` column is stored as an **object**, which usually means text in pandas.

For time-based analysis, we want dates to be stored in pandas' proper **datetime** format instead. This makes it easier to sort dates, filter by time period, and perform date-based analysis later on.

In this step, we will use `pd.to_datetime()` to convert the `Delivery Date` column from an object to a datetime data type.

This is a common example of **data cleaning** in pandas, because we are changing a column into the correct format for analysis.


In [5]:
# Change Delivery Date values to datetime
df['Delivery Date'] = pd.to_datetime(df['Delivery Date'])

# Confirm that date values is now in datetime format
df.dtypes

Delivery Date            datetime64[ns]
Delivery Hour                     int64
Settlement Point Name            object
Average Hourly Price            float64
dtype: object

## Step 4: Create a Month Column

Now that the `Delivery Date` column is in datetime format, we can start extracting useful pieces of information from it.

In this step, we will create a new column called `Month` using the `.dt.month` attribute.

This will give us the month number for each row, which can be useful later for grouping, filtering, and analyzing the data by month.


In [6]:
# Create a new column called Month using the Delivery Date column
df['Month'] = df['Delivery Date'].dt.month

# Check to see if Month column has been added
df.head()

,Delivery Date,Delivery Hour,Settlement Point Name,Average Hourly Price,Month
0,2025-01-01,1,HB_HOUSTON,18.0775,1
1,2025-01-01,1,HB_NORTH,18.0775,1
2,2025-01-01,1,HB_PAN,18.0775,1
3,2025-01-01,1,HB_SOUTH,18.0775,1
4,2025-01-01,1,HB_WEST,18.0775,1


## Step 5: Boolean Mask

Sometimes, we do not want to analyze the entire dataset at once. Instead, we may want to focus on just one settlement point.

In this step, we will use a **Boolean mask** to create a new DataFrame that only contains rows where the `Settlement Point Name` is `HB_HOUSTON`, which represents the Houston Hub.

This is a common pandas technique for filtering data based on a condition.

In [7]:
# Create a Boolean mask for Houston Hub
mask = df['Settlement Point Name'] == 'HB_HOUSTON'

# Use the mask to create a new DataFrame
houston_df = df[mask].copy()

# Display the first few rows
houston_df.head()

,Delivery Date,Delivery Hour,Settlement Point Name,Average Hourly Price,Month
0,2025-01-01,1,HB_HOUSTON,18.0775,1
5,2025-01-01,2,HB_HOUSTON,17.6875,1
10,2025-01-01,3,HB_HOUSTON,17.0950,1
15,2025-01-01,4,HB_HOUSTON,17.0050,1
20,2025-01-01,5,HB_HOUSTON,18.5750,1


## Step 6: Groupby

Now that we have isolated Houston Hub data in `houston_df`, we can start summarizing it.

In this step, we will use `groupby()` to calculate the **average price** for each combination of `Month` and `Delivery Hour`.

This allows us to see how Houston Hub prices vary depending on the month of the year and the hour of the day.

Grouping data like this is a very common pandas technique for summarizing and analyzing patterns in a dataset.


In [8]:
# Group Houston Hub data by Month and Delivery Hour
# Then calculate the average price for each group
houston_month_hour_avg = houston_df.groupby(
    ['Month', 'Delivery Hour']
)['Average Hourly Price'].mean().reset_index()

# Display the first few rows
houston_month_hour_avg.head()

,Month,Delivery Hour,Average Hourly Price
0,1,1,26.901613
1,1,2,26.493952
2,1,3,26.279839
3,1,4,26.954355
4,1,5,28.808710


## Step 7: Create and Round a 12 x 24 Pivot Table

Now that we have calculated the average Houston Hub price for each combination of `Month` and `Delivery Hour`, we can reorganize the data into a more readable table.

In this step, we will use `.pivot()` to create a new DataFrame where:

* each row represents a month
* each column represents a delivery hour
* each value represents the average Houston Hub price

Then, we will round the values to make the table easier to read.

This gives us a clean **12 x 24 pivot table**, which is the final goal of this project.

In [ ]:
# Create a 12 x 24 pivot table from the grouped Houston Hub data
houston_pivot = houston_month_hour_avg.pivot(
    index='Month',
    columns='Delivery Hour',
    values='Average Hourly Price'
)

# Round the pivot table to 2 decimal places
houston_pivot = houston_pivot.round(2)

# Display the final pivot table
houston_pivot

## Congratulations!

You just completed a **real Python/pandas assessment** that is commonly used in interviews for **full-time Structured Power Trading** roles.

In other words, you just worked through the same type of task that firms may use to test whether a candidate can take raw power market data, manipulate it correctly, and turn it into something useful for analysis.

